# EnerGNN: hands-on overview

*Notebook of the EnerGNN webinar of 18 November 2026: [recording](https://youtu.be/g6Ub_gZYslQ?t=40) (YouTube), [slides](energnn_webinar.pdf) (PDF).*

A GNN library on [JAX](https://docs.jax.dev/) / [Flax](https://flax.readthedocs.io/) for real-life energy networks.

In this webinar we'll showcase four key aspects of `energnn`:
1. **Data** (`energnn.graph`): networks as *Hyper Heterogeneous Multi Graphs* (H2MG)
2. **Problem** (`energnn.problem`): a use case = contexts, decisions, a score and a gradient (here: voltage control on PEGASE 1354)
3. **Models** (`energnn.model`): ready-to-use GNNs and their building blocks
4. **Training** (`energnn.trainer`): how to train a model for your own problem.

In [1]:
import copy
import logging
import os
import warnings

os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
warnings.filterwarnings("ignore")
logging.getLogger("powsybl").setLevel(logging.ERROR)

import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd

## 1. Data

Let us start with the core data representation module, `energnn.graph`.

<p align="center">
  <img src="../../_static/energnn_h2mg_black.png" alt="H2MG" width="800" class="only-light" style="display: block; margin: 0 auto; background: white;">
  <img src="../../_static/energnn_h2mg_white.png" alt="H2MG" width="800" class="only-dark" style="display: none;">
</p>

**EnerGNN** graphs are **not standard graphs**, but **Hyper Heterogeneous Multi Graphs** (H2MGs):
- **Hyper**: an object may connect any number of addresses (one for a load, two for a line, three for a three-winding transformer), not just two.
- **Heterogeneous**: objects belong to different classes (lines, loads, generators...), each with its own ports and features.
- **Multi**: several objects may connect the same addresses (parallel lines, several loads at one bus).


**EnerGNN** graphs are made of:
- **Objects** (hyper-edges), which are grouped by **class**; a class has common **ports** and **features**.
- **Addresses** which are the glue between objects: objects whose ports hold the same address are connected.

### 1.1. Loading and saving

In [2]:
from energnn.graph import Graph

graph = Graph.from_pickle(file_path="ieee14.pkl")  # the IEEE 14-bus network
graph.to_pickle("/tmp/my_graph.pkl")

### 1.2. Printing

In [3]:
print(graph)

Graph · 6 hyper-edge sets · 14 addresses
│
├─ bus · 14 objects · 1 port · 1 feature
│     │ ports │ features
│     │    id │    v_mag
│  ───┼───────┼─────────
│   0 │     5 │  1.06000
│   1 │     6 │  1.04500
│   2 │     7 │  1.01000
│   3 │     8 │  1.01767
│   4 │     9 │  1.01951
│   5 │    10 │  1.07000
│   6 │    11 │  1.06152
│   7 │    12 │  1.09000
│   8 │    13 │  1.05593
│   9 │     0 │  1.05098
│  10 │     1 │  1.05691
│  11 │     2 │  1.05519
│  12 │     3 │  1.05038
│  13 │     4 │  1.03553
│
├─ generator · 5 objects · 2 ports · 2 features
│    │          ports           │      features
│    │ bus_id  regulated_bus_id │ target_p  target_v
│  ──┼──────────────────────────┼───────────────────
│  0 │      5                 5 │    2.324     1.060
│  1 │      6                 6 │    0.400     1.045
│  2 │      7                 7 │    0.000     1.010
│  3 │     10                10 │    0.000     1.070
│  4 │     12                12 │    0.000     1.090
│
├─ line · 17 objects

### 1.3. Plotting

In [4]:
import json

import pandapower.networks

from energnn.graph.visualization import plot_graph

# pandapower ships coordinates for the IEEE 14 buses: stored as features of the "bus" class, they place its objects.
graph.bus.x, graph.bus.y = np.array([json.loads(geo)["coordinates"] for geo in pandapower.networks.case14().bus.geo]).T

plot_graph(graph, hyper_edge_positions={"bus": ["x", "y"]}, size=800, pull_coeff=30, marker_size=0.3,
           repulsion_coeff=10)  # hover, zoom, click the legend

### 1.4. Adding features

In [5]:
graph.load.s0 = np.hypot(graph.load.p0, graph.load.q0)
print(graph.load)

HyperEdgeSet · 11 objects · 1 port · 3 features
   │ ports  │         features
   │ bus_id │    p0      q0         s0
───┼────────┼─────────────────────────
 0 │      6 │ 0.217   0.127  0.2514319
 1 │      7 │ 0.942   0.190  0.9609703
 2 │      8 │ 0.478  -0.039  0.4795884
 3 │      9 │ 0.076   0.016  0.0776659
 4 │     10 │ 0.112   0.075  0.1347924
 5 │     13 │ 0.295   0.166  0.3384981
 6 │      0 │ 0.090   0.058  0.1070701
 7 │      1 │ 0.035   0.018  0.0393573
 8 │      2 │ 0.061   0.016  0.0630635
 9 │      3 │ 0.135   0.058  0.1469320
10 │      4 │ 0.149   0.050  0.1571655


### 1.5. Building a graph

There are four ways to build an **EnerGNN** graph:
1. From scratch;
2. Using your own data conversion pipeline;
3. Using the `pypowsybl-to-energnn` converter;
4. Using the `pandapower-to-energnn` converter.

#### 1.5.1. Building from scratch

In [6]:
from energnn.graph import HyperEdgeSet

graph_from_scratch = Graph.from_dict(
    hyper_edge_set_dict={
        "bus": HyperEdgeSet.from_dict(
            port_dict={"id": np.array([0, 1, 2])},
            feature_dict={"p": np.array([1.0, -0.4, -0.6])},
        ),
        "line": HyperEdgeSet.from_dict(
            port_dict={"from": np.array([0, 0]), "to": np.array([1, 2])},
            feature_dict={"x": np.array([0.1, 0.2])},
        ),
    },
    n_addresses=3,
)

In [7]:
plot_graph(graph_from_scratch, size=300)

In [8]:
print(graph_from_scratch)

Graph · 2 hyper-edge sets · 3 addresses
│
├─ bus · 3 objects · 1 port · 1 feature
│    │ ports │ features
│    │    id │        p
│  ──┼───────┼─────────
│  0 │     0 │      1.0
│  1 │     1 │     -0.4
│  2 │     2 │     -0.6
│
└─ line · 2 objects · 2 ports · 1 feature
     │  ports   │ features
     │ from  to │        x
   ──┼──────────┼─────────
   0 │    0   1 │      0.1
   1 │    0   2 │      0.2


#### 1.5.2. Building using your own converter

In [9]:
from energnn.converter import Converter, ElementsConverter

raw_tables = {
    "bus": pd.DataFrame({"id": ["Paris", "Lyon", "Lille"]}),
    "line": pd.DataFrame({"from": ["Paris", "Paris", "Paris"], "to": ["Lyon", "Lyon", "Lille"], "x": [0.12, 0.12, 0.10]}),
    "generator": pd.DataFrame({"bus": ["Lyon", "Lille"], "p": [900.0, 600.0], "fuel": ["nuclear", "gas"]}),
    "load": pd.DataFrame({"bus": ["Paris", "Lille"], "p": [1200.0, 300.0]}),
}


class MyTableConverter(ElementsConverter):
    def __init__(self, name, port_list, feature_list):
        super().__init__(port_list, feature_list)
        self.name = name

    def _get_table(self, tables):
        return tables[self.name]


class MyConverter(Converter):
    elements_converter_dict = {
        "bus": MyTableConverter("bus", ["id"], None),
        "line": MyTableConverter("line", ["from", "to"], ["x"]),
        "generator": MyTableConverter("generator", ["bus"], ["p", "fuel"]),
        "load": MyTableConverter("load", ["bus"], ["p"]),
    }


my_converter = MyConverter()
graph_from_my_converter = my_converter(raw_tables)

In [10]:
plot_graph(graph_from_my_converter, size=300)

In [11]:
print(graph_from_my_converter)

Graph · 4 hyper-edge sets · 3 addresses
│
├─ bus · 3 objects · 1 port · 0 features
│    │ ports
│    │    id
│  ──┼──────
│  0 │     2
│  1 │     1
│  2 │     0
│
├─ generator · 2 objects · 1 port · 2 features
│    │ ports │    features
│    │   bus │     fuel    p
│  ──┼───────┼──────────────
│  0 │     1 │ 0.302416  900
│  1 │     0 │ 0.933736  600
│
├─ line · 3 objects · 2 ports · 1 feature
│    │  ports   │ features
│    │ from  to │        x
│  ──┼──────────┼─────────
│  0 │    2   1 │     0.12
│  1 │    2   1 │     0.12
│  2 │    2   0 │     0.10
│
└─ load · 2 objects · 1 port · 1 feature
     │ ports │ features
     │   bus │        p
   ──┼───────┼─────────
   0 │     2 │     1200
   1 │     0 │      300


#### 1.5.3. Using the [pypowsybl-to-energnn](https://github.com/energnn/pypowsybl-to-energnn)

Companion repo for importing **EnerGNN** graphs from **PyPowSyBl** networks.

In [12]:
import pypowsybl.loadflow as lf
import pypowsybl.network as pn
import pypowsybl_to_energnn as pe

pypowsybl_converter = pe.PypowsyblConverter({
    "bus": pe.Buses(features=("v_mag",)),
    "line": pe.Lines(features=("r", "x", "b1", "b2", "connected1", "connected2")),
    "transformer": pe.TwoWindingsTransformers(features=("r", "x", "rho", "connected1", "connected2")),
    "generator": pe.Generators(features=("target_p", "target_v", "q")),
    "load": pe.Loads(features=("p0", "q0")),
    "shunt": pe.ShuntCompensators(features=("b",)),
})

pypowsybl_network = pn.load("case1354pegase.mat")
pypowsybl_graph = pypowsybl_converter(network=pypowsybl_network)

In [13]:
# Get bus coordinates to improve readability: pandapower ships (synthetic) ones for the PEGASE buses,
# and bus n of pandapower is "BUS-(n+1)" in pypowsybl.
pp_bus = pandapower.networks.case1354pegase().bus
BUS_XY = pd.DataFrame(
    [json.loads(geo)["coordinates"] for geo in pp_bus.geo],
    index=pypowsybl_network.get_bus_breaker_view_buses().bus_id.loc["BUS-" + (pp_bus.name + 1).astype(str)])
pypowsybl_graph.bus.x, pypowsybl_graph.bus.y = BUS_XY.loc[pypowsybl_network.get_buses().index].to_numpy().T

plot_graph(pypowsybl_graph, size=800, marker_size=0.2, hyper_edge_positions={"bus": ["x", "y"]}, repulsion_coeff=0.3,
           pull_coeff=30, iterations=15)

In [14]:
print(pypowsybl_graph)

Graph · 6 hyper-edge sets · 1354 addresses
│
├─ bus · 1354 objects · 1 port · 3 features
│       │ ports │            features
│       │    id │    v_mag          x          y
│  ─────┼───────┼───────────────────────────────
│     0 │   556 │ 1.016674   1.270954  12.133555
│     1 │   712 │ 1.026472   9.299412   4.391116
│     2 │   713 │ 1.030298   8.348078   4.172749
│     3 │    20 │ 1.038534  -0.553911  -5.440670
│     4 │   235 │ 1.036195  -4.793856  -0.219752
│     5 │   251 │ 1.035058   0.850696   2.903424
│     6 │   309 │ 1.032267   5.392877   5.683593
│     7 │   371 │ 0.994409   3.881266   8.118519
│     8 │   638 │ 1.045829  -0.446954  -1.138800
│     9 │   639 │ 0.990542  -0.660558  -1.044981
│    10 │   640 │ 0.984486  -0.414036  -0.445682
│    11 │   641 │ 1.050811  -0.668389  -1.448275
│    12 │   774 │ 1.063422  14.990235   5.630857
│    13 │   849 │ 0.983702   5.446055  13.371058
│    14 │   850 │ 0.988112   4.891432  12.119894
│   ⋯   │   ⋯   │               ⋯
│  133

#### 1.5.4. Using the [pandapower-to-energnn](https://github.com/energnn/pandapower-to-energnn)

Companion repo for importing **EnerGNN** graphs from **PandaPower** networks.

In [15]:
import pandapower_to_energnn as ppe

pandapower_converter = ppe.ACLoadFlowInputConverter()
pandapower_network = pandapower.networks.case118()  # the IEEE 118-bus network
pandapower_graph = pandapower_converter(net=pandapower_network)

In [16]:
# pandapower ships coordinates for this network too.
pandapower_graph.buses.x, pandapower_graph.buses.y = np.array(
    [json.loads(geo)["coordinates"] for geo in pandapower_network.bus.geo]).T

plot_graph(pandapower_graph, size=800, marker_size=0.3, hyper_edge_positions={"buses": ["x", "y"]}, repulsion_coeff=10,
           pull_coeff=30)

In [17]:
print(pandapower_graph)

Graph · 8 hyper-edge sets · 118 addresses
│
├─ buses · 118 objects · 1 port · 2 features
│      │     ports      │        features
│      │ energnn_adress │          x          y
│  ────┼────────────────┼──────────────────────
│    0 │              0 │  -2.275371   2.854341
│    1 │              1 │  -2.936819   2.212179
│    2 │              2 │  -1.834431   1.709445
│    3 │              3 │  -0.888627   1.553271
│    4 │              4 │  -0.963283   0.694730
│    5 │              5 │  -0.410973   1.767572
│    6 │              6 │  -1.483650   2.262042
│    7 │              7 │  -0.679300  -0.581321
│    8 │              8 │   0.492210   0.039545
│    9 │              9 │   1.508649   0.753799
│   10 │             10 │  -1.808481   1.078349
│   11 │             11 │  -2.452116   1.188777
│   12 │             12 │  -2.533989   0.288249
│   13 │             13 │  -3.086774   0.340720
│   14 │             14 │  -2.329747  -0.679471
│   ⋯  │       ⋯        │           ⋯
│  103 │       

### 1.6. Padding and batching

Networks differ in size (outages, maintenance...). To stack them into a **batch**, the smaller ones are **padded**
with *fictitious* objects (`×`), which models and scores ignore.

In [18]:
from energnn.graph import collate_graphs, max_shape, separate_graphs

raw_tables_outage = {**raw_tables, "line": raw_tables["line"].drop(index=1), "load": raw_tables["load"].drop(index=1)}
graph_1 = my_converter(raw_tables)
graph_2 = my_converter(raw_tables_outage)
graphs = [graph_1, graph_2]

In [19]:
plot_graph(graph_1, size=300)

In [20]:
plot_graph(graph_2, size=300)

In [21]:
target_shape = max_shape([g.true_shape for g in graphs])
for g in graphs:
    g.pad(target_shape)
batch = collate_graphs(graphs)  # separate_graphs(batch) and g.unpad() go the other way

print(batch)

Graph · batch of 2 · 4 hyper-edge sets · 3 addresses
│
├─ bus · batch of 2 · 3 objects · 1 port · 0 features
│             │ ports
│  batch  obj │    id
│  ───────────┼──────
│      0    0 │     2
│      0    1 │     1
│      0    2 │     0
│      1    0 │     2
│      1    1 │     1
│      1    2 │     0
│
├─ generator · batch of 2 · 2 objects · 1 port · 2 features
│             │ ports │    features
│  batch  obj │   bus │     fuel    p
│  ───────────┼───────┼──────────────
│      0    0 │     1 │ 0.302416  900
│      0    1 │     0 │ 0.933736  600
│      1    0 │     1 │ 0.302416  900
│      1    1 │     0 │ 0.933736  600
│
├─ line · batch of 2 · 3 objects · 2 ports · 1 feature
│                │  ports   │ features
│  batch  obj    │ from  to │        x
│  ──────────────┼──────────┼─────────
│      0    0    │    2   1 │     0.12
│      0    1    │    2   1 │     0.12
│      0    2    │    2   0 │     0.10
│      1    0    │    2   1 │     0.12
│      1    1    │    2   0 │     0.1

### 1.7. Backends

Graphs are built on **NumPy** (shapes vary freely), then moved to **JAX** once padded: a `Graph` is a pytree, so it
goes through `jit`, `vmap` and `grad`.

In [22]:
from energnn.graph import JaxBackend

jax_batch = batch.to_backend(JaxBackend())
print(type(batch.line.x).__name__, "->", type(jax_batch.line.x).__name__)

ndarray -> ArrayImpl


## 2. Problem

Let us now see how a use case is plugged into EnerGNN, with `energnn.problem`.

### 2.1. A concrete use case: voltage control

- **Objective**: Keep all voltages within **[0.95, 1.05] p.u.**.
- **Levers**: Voltage setpoints of generators.

We want a GNN that takes as input a power grid operating condition, and returns voltage setpoints.

Let's write the use case with plain pypowsybl first, by answering four questions. Then we'll show how to use EnerGNN for it.

In [23]:
# Plot helpers (not the point)

import plotly.graph_objects as go

# BUS_XY, the coordinates of the PEGASE buses, comes from section 1.
# Same layout settings as the PEGASE plot of section 1.
PLOT = {"size": 800, "marker_size": 0.2, "repulsion_coeff": 0.3, "pull_coeff": 30, "iterations": 15}
# plot_graph draws 16 shades, evenly spaced between the lowest and the highest of these values: here every 0.02 from
# 0.86 to 1.16, so that a shade changes exactly at 0.95 and at 1.05. Dark to medium blue below the band
# (undervoltage), pale blue to pale red inside, bright to dark red above (overvoltage).
UNDER, OVER = "#4292c6", "#ef3b2c"
VOLTAGE_COLORS = {0.86: "#08306b", 0.94: UNDER, 0.96: "#c6dbef", 1.0: "#e0e0e0", 1.04: "#fcbba1", 1.06: OVER,
                  1.16: "#67000d"}


def plot_voltages(context, network, v_mag=None, address_positions=None):
    """The graph with its buses colored by voltage (`v_mag` replaces the voltages of the context).
    By default the bus objects are placed on the map; `address_positions` places the addresses instead."""
    context = copy.deepcopy(context)
    if v_mag is not None:
        context.bus.v_mag = np.asarray(v_mag)
    if address_positions is None:
        context.bus.x, context.bus.y = BUS_XY.loc[network.get_buses().index].to_numpy().T
        positions = {"hyper_edge_positions": {"bus": ["x", "y"]}}
    else:
        positions = {"address_positions": address_positions}
    return plot_graph(context, hyper_edge_colors={"bus": "v_mag"}, values_to_colors=VOLTAGE_COLORS, **positions, **PLOT)


def n_violations(v):
    return int(((np.asarray(v) < 0.95) | (np.asarray(v) > 1.05)).sum())


def voltage_histogram(voltages):
    "Distribution of the bus voltages, one trace per named array."
    fig = go.Figure()
    for (name, v), color in zip(voltages.items(), ["#898781", "#1f8ff0", "#f0883a"]):
        fig.add_histogram(x=v, name=name, xbins={"size": 0.005}, marker={"color": color}, opacity=0.6)
    for limit in (0.95, 1.05):
        fig.add_vline(x=limit, line={"color": "#0b0b0b", "width": 1, "dash": "dot"})
    low, high = min(np.nanmin(v) for v in voltages.values()), max(np.nanmax(v) for v in voltages.values())
    fig.add_vrect(x0=min(low, 0.95) - 0.005, x1=0.95, fillcolor=UNDER, opacity=0.12, line_width=0, layer="below")
    fig.add_vrect(x0=1.05, x1=max(high, 1.05) + 0.005, fillcolor=OVER, opacity=0.12, line_width=0, layer="below")
    fig.update_layout(barmode="overlay", xaxis_title="bus voltage (p.u.)", yaxis_title="buses", width=720, height=320,
                      margin={"l": 60, "r": 20, "t": 20, "b": 50}, plot_bgcolor="white", legend={"x": 0.01, "y": 0.99})
    return fig

#### ① What do we know? → the **context** (*i.e.* the model input)

The network and its state at the current setpoints: one power flow, one conversion (the `pypowsybl_converter` of section 1).

In [24]:
LF_PARAMETERS = lf.Parameters(use_reactive_limits=False, provider_parameters={"networkCacheEnabled": "true"})
V_MIN, V_MAX = 0.95, 1.05


def get_context(network):
    lf.run_ac(network, parameters=LF_PARAMETERS)
    return pypowsybl_converter(network=network)


context = get_context(pypowsybl_network)

print(n_violations(context.bus.v_mag), "buses out of the band")
plot_voltages(context, pypowsybl_network)

385 buses out of the band


#### ② What do we decide? → the **decision**

A relative change `delta_v` of the setpoint of each generator.

It is a graph too:
- it has a single class (`generator`),
- with no port (the connectivity is already in the context),
- with one feature (`delta_v`).

In [25]:
from energnn.graph import GraphStructure, HyperEdgeSetStructure

DECISION_STRUCTURE = GraphStructure(
    hyper_edge_sets={"generator": HyperEdgeSetStructure(port_list=None, feature_list=["delta_v"])})


def to_decision(delta_v):
    return Graph.from_dict(hyper_edge_set_dict={"generator": HyperEdgeSet.from_dict(feature_dict={"delta_v": delta_v})},
                           n_addresses=0)


n_generators = len(context.generator.target_v)
print(to_decision(np.zeros(n_generators)))  # one setpoint change per generator of the context

Graph · 1 hyper-edge set · 0 addresses
│
└─ generator · 260 objects · 0 ports · 1 feature
       │ features
       │  delta_v
   ────┼─────────
     0 │        0
     1 │        0
     2 │        0
     3 │        0
     4 │        0
     5 │        0
     6 │        0
     7 │        0
     8 │        0
     9 │        0
    10 │        0
    11 │        0
    12 │        0
    13 │        0
    14 │        0
    ⋯  │    ⋯
   245 │        0
   246 │        0
   247 │        0
   248 │        0
   249 │        0
   250 │        0
   251 │        0
   252 │        0
   253 │        0
   254 │        0
   255 │        0
   256 │        0
   257 │        0
   258 │        0
   259 │        0


#### ③ How good is a decision? → the **score**

Apply the setpoints, re-run the power flow, measure over voltages and under voltage.

Any code will do: here the OpenLoadFlow simulator.

In [26]:
REFERENCE_V = pypowsybl_network.get_generators().target_v  # reference setpoints (kV), per generator
NOMINAL_V = pypowsybl_network.get_voltage_levels().nominal_v.loc[pypowsybl_network.get_buses().voltage_level_id].set_axis(
    pypowsybl_network.get_buses().index)
# Same power flow, other parameters: running it makes OpenLoadFlow drop its cache for the network.
LF_RESET = lf.Parameters(use_reactive_limits=False,
                         provider_parameters={"networkCacheEnabled": "true", "maxNewtonRaphsonIterations": "30"})


def simulate(network, delta_v):
    "Apply the setpoint changes (clipped to ±10 %), run the power flow, return the bus voltages in p.u."
    delta_v = np.clip(np.asarray(delta_v), -0.1, 0.1)
    network.update_generators(id=REFERENCE_V.index, target_v=REFERENCE_V * (1 + delta_v))
    result = lf.run_ac(network, parameters=LF_PARAMETERS)
    if result[0].status != lf.ComponentStatus.CONVERGED:  # bad warm start: drop the solver cache, retry from a flat start
        lf.run_ac(network, parameters=LF_RESET)
        result = lf.run_ac(network, parameters=LF_PARAMETERS)
    v = (network.get_buses().v_mag / NOMINAL_V).to_numpy()
    if result[0].status != lf.ComponentStatus.CONVERGED:
        v[:] = np.nan
    return v, delta_v


def voltage_objective(v, delta_v):
    "Squared voltage violations (in %), plus a small penalty on the setpoint changes."
    violation = 100 * (jnp.maximum(v - V_MAX, 0) + jnp.maximum(V_MIN - v, 0))
    return jnp.sum(violation ** 2) + 10.0 * jnp.sum(delta_v ** 2)


def get_score(network, delta_v):
    return float(voltage_objective(*simulate(network, delta_v)))


print("score with no change:", round(get_score(pypowsybl_network, np.zeros(len(REFERENCE_V))), 1))

score with no change: 1030.4


#### ④ How to improve a decision? → the **gradient**

The derivative of the objective with regards to the decision.

Here there is **no label** and no loss written in JAX: the chain rule goes through the power flow, thanks to the
*adjoint* of OpenLoadFlow (`run_adjoint`, one linear solve).

In [27]:
import pypowsybl as pp
from pypowsybl.sensitivity import ContingencyContextType, SensitivityFunctionType, SensitivityVariableType


def get_gradient(network, delta_v):
    """Derivative of the objective w.r.t. the lever delta_v, computed by differentiating through the simulator."""
    v, delta_v = simulate(network, delta_v)
    dL_dv, dL_ddelta = jax.grad(voltage_objective, argnums=(0, 1))(v, delta_v)  # objective: plain JAX
    analysis = pp.sensitivity.create_ac_analysis()  # power flow: adjoint, dL/dv -> dL/dtarget_v
    analysis.add_factor_matrix(list(NOMINAL_V.index), list(REFERENCE_V.index), [], ContingencyContextType.NONE,
                               SensitivityFunctionType.BUS_VOLTAGE, SensitivityVariableType.BUS_TARGET_VOLTAGE)
    result = analysis.run_adjoint(network, cotangents=np.asarray(dL_dv) / NOMINAL_V.to_numpy(), parameters=LF_PARAMETERS)
    dL_dtarget_v = result.get_gradient().reindex(REFERENCE_V.index).fillna(0).to_numpy()
    return np.nan_to_num(dL_dtarget_v * REFERENCE_V.to_numpy() + np.asarray(dL_ddelta))

### 2.2. From the use case to the EnerGNN interface

The training loop only needs the pieces we just wrote:

```python
# Training
for batch in train_loader:                         # situations x ~ p(x)
    x = batch.get_context()                        # context     x
    y = model(x)                                   # decision    ŷ = ŷ_θ(x)
    g = batch.get_gradient(y)                      # gradient    g = ∇_y f(ŷ; x)
    theta = optimizer_step(theta, vjp(model, g))   # backprop    θ ← θ − η (∂ŷ_θ/∂θ)ᵀ g
```

> **The chain rule has a natural seam. We put the interface there.**
>
> $\nabla_\theta f = (\partial \hat{y}_\theta / \partial \theta)^\top \, \nabla_y f$: the right factor is yours, the left one is EnerGNN's.

**Why this split?**

- **You give a gradient, not a loss**: *EnerGNN never asks what your problem is. It only asks which way each output
  should move.* Your simulator doesn't need to speak JAX. It only needs to answer one question. Supervised learning
  is the special case $\nabla_y f = \hat{y} - y^{label}$.
- **Unusual in deep learning, standard in optimisation**: IPOPT asks for gradients for instance.
- **The score is separate from the gradient**: what you report (number of violations, a cost) need not be
  differentiable, and the gradient may come from a smooth surrogate.

The `Problem` is a thin wrapper around our functions:

In [28]:
from energnn.problem import Problem


class VoltageControlProblem(Problem):
    "One instance = one variant of the pypowsybl network."

    def __init__(self, network, variant):
        self.network, self.variant = network, variant
        self.context = get_context(self.select())

    def select(self):
        self.network.set_working_variant(self.variant)
        return self.network

    context_structure = property(lambda self: pypowsybl_converter.get_structure())
    decision_structure = property(lambda self: DECISION_STRUCTURE)

    def get_context(self, step_with_metrics=False, step=None):
        return self.context, {}

    def get_score(self, *, decision, step_with_metrics=False, step=None):
        return get_score(self.select(), decision.generator.delta_v), {}

    def get_gradient(self, *, decision, step_with_metrics=False, step=None):
        gradient = copy.deepcopy(decision)  # a gradient has the structure of the decision
        gradient.generator.delta_v = get_gradient(self.select(), decision.generator.delta_v)
        return gradient, {}

    def save(self, *, path):
        self.network.save(path)


problem = VoltageControlProblem(pypowsybl_network, "InitialState")
print("score with no change:", round(problem.get_score(decision=to_decision(np.zeros(len(REFERENCE_V))))[0], 1))

score with no change: 1030.4


#### Batch and loader, in short

Training needs many situations, several at a time. Two more classes, with **no business logic**:

- **`ProblemBatch`**: a few problems solved together. Their contexts are **padded and stacked** (section 1.6) into one
  batched graph for the GNN; the batched decision is split back, and each problem computes its own score and gradient
  (this is where a real use case would run its simulators in parallel).
- **`ProblemLoader`**: iterates over a dataset and yields batches; one for training, one for validation.

**The dataset**, concretely: variants of the PEGASE network, each one wrapped in a `VoltageControlProblem`.
Loads are rescaled at random by ±20 % and, half of the time, one line is taken out; a variant is kept if its
power flow converges with every bus connected. 128 variants for training, 32 for validation.

In [29]:
from energnn.problem import ProblemBatch, ProblemLoader


def to_batch(graphs):
    "Pad to a common shape, stack, move to JAX (section 1.6)."
    graphs = copy.deepcopy(graphs)
    shape = max_shape([g.true_shape for g in graphs])
    for g in graphs:
        g.pad(shape)
    return collate_graphs(graphs).to_backend(JaxBackend())


def from_batch(graph):
    "The other way: one NumPy graph per problem, without its fictitious objects."
    graphs = separate_graphs(graph.to_numpy_backend())
    for g in graphs:
        g.unpad()
    return graphs


class VoltageControlBatch(ProblemBatch):
    context_structure, decision_structure = pypowsybl_converter.get_structure(), DECISION_STRUCTURE

    def __init__(self, problems):
        self.problems = problems

    def get_context(self, **kwargs):
        return to_batch([p.context for p in self.problems]), {}

    def get_score(self, *, decision, **kwargs):
        return [p.get_score(decision=d)[0] for p, d in zip(self.problems, from_batch(decision))], {}

    def get_gradient(self, *, decision, **kwargs):
        return to_batch([p.get_gradient(decision=d)[0] for p, d in zip(self.problems, from_batch(decision))]), {}


class VoltageControlLoader(ProblemLoader):
    context_structure, decision_structure = pypowsybl_converter.get_structure(), DECISION_STRUCTURE

    def __init__(self, problems, batch_size):
        self.problems, self.batch_size = problems, batch_size

    def __iter__(self):
        self.starts = iter(range(0, len(self.problems), self.batch_size))
        return self

    def __next__(self):
        start = next(self.starts)  # StopIteration ends the epoch
        return VoltageControlBatch(self.problems[start:start + self.batch_size])

    def __len__(self):
        return len(range(0, len(self.problems), self.batch_size))

In [30]:
def make_dataset(network, n_problems, seed):
    "Random variants of the network whose reference power flow converges with every bus connected."
    rng = np.random.default_rng(seed)
    problems = []
    while len(problems) < n_problems:
        variant = f"{seed}-{len(problems)}"
        network.clone_variant("InitialState", variant)
        network.set_working_variant(variant)
        network.update_generators(id=REFERENCE_V.index, target_v=REFERENCE_V)  # start from the reference setpoints
        loads = network.get_loads()
        factor = rng.uniform(0.8, 1.2, size=len(loads))
        network.update_loads(id=loads.index, p0=loads.p0 * factor, q0=loads.q0 * factor)
        if rng.random() < 0.5:
            network.update_lines(id=rng.choice(network.get_lines().index), connected1=False, connected2=False)
        converged = lf.run_ac(network, parameters=LF_PARAMETERS)[0].status == lf.ComponentStatus.CONVERGED
        if converged and (network.get_buses().connected_component == 0).all():
            problems.append(VoltageControlProblem(network, variant))
        else:
            network.remove_variant(variant)
    return problems


train_loader = VoltageControlLoader(make_dataset(pypowsybl_network, 128, seed=7), batch_size=8)
val_loader = VoltageControlLoader(make_dataset(pypowsybl_network, 32, seed=8), batch_size=8)
context_batch, _ = next(iter(val_loader)).get_context()
print(len(train_loader), "training batches,", len(val_loader), "validation batches")
print("generator features (batch, objects, features):", context_batch.generator.feature_array.shape)

16 training batches, 4 validation batches
generator features (batch, objects, features): (8, 260, 3)


## 3. Models

Now that the problem is defined, let us build a model to solve it, with `energnn.model`.

Models are composed of four blocks:
1. **Normalizer**: online feature statistics (`TDigestNormalizer` handles multi-modal and multi-scale feature distributions);
2. **Encoder**: one MLP per class;
3. **Coupler**: embeds addresses into a latent space, (through message passing or something else);
4. **Decoder**: one output per object (*equivariant*) or one per graph (*invariant*).

Our GNN models are equivariant per permutation: permuting input data results in a permutation over the output.

### 3.1. Ready-to-use models
Predefined architectures are available for quick trials: `Tiny`, `Small`, `Medium`, `Large`, `ExtraLarge` `RecurrentEquivariantGNN`.

To build them, you just need to tell them the input structure, and the output structure.

In [31]:
from flax import nnx

from energnn.model.ready_to_use import MediumRecurrentEquivariantGNN

in_structure, out_structure = train_loader.context_structure, train_loader.decision_structure

print("input structure (what features the model takes as input, on which object class):\n", in_structure)
print("output structure (what features the model predicts, on which object class):\n", out_structure)

input structure (what features the model takes as input, on which object class):
 GraphStructure · 6 hyper-edge sets
├─ bus           ports: id                         features: v_mag
├─ line          ports: bus1_id, bus2_id           features: r, x, b1, b2, connected1, connected2
├─ transformer   ports: bus1_id, bus2_id           features: r, x, rho, connected1, connected2
├─ generator     ports: bus_id, regulated_bus_id   features: target_p, target_v, q
├─ load          ports: bus_id                     features: p0, q0
└─ shunt         ports: bus_id                     features: b
output structure (what features the model predicts, on which object class):
 GraphStructure · 1 hyper-edge set
└─ generator   ports: —   features: delta_v


In [32]:
model = MediumRecurrentEquivariantGNN(in_structure=train_loader.context_structure,
                                      out_structure=train_loader.decision_structure)
model.eval()  # train mode updates the normalizer statistics


def n_params(model):
    return sum(int(np.prod(leaf.shape)) for leaf in jax.tree.leaves(nnx.state(model, nnx.Param)))


print("trainable parameters:", n_params(model))

trainable parameters: 18433


### 3.2. Your own model

Any combination of blocks is a `GNN`: here attention-based message passing and a center-reduce normalizer.
This is easily compatible with

In [33]:
from energnn.model import (
    GNN,
    MLP,
    CenterReduceNormalizer,
    GATv2MessagePassingFunction,
    MLPEncoder,
    MLPEquivariantDecoder,
    RecurrentCoupler,
)

latent, rngs = 8, nnx.Rngs(0)

custom_model = GNN(
    normalizer=CenterReduceNormalizer(in_structure=in_structure, update_limit=1000),
    encoder=MLPEncoder(in_structure=in_structure, hidden_sizes=[16], out_size=latent, rngs=rngs),
    coupler=RecurrentCoupler(
        phi=MLP(in_size=latent, hidden_sizes=[], out_size=latent, final_activation=nnx.tanh, rngs=rngs),
        message_functions=[
            GATv2MessagePassingFunction(
                in_graph_structure=in_structure, in_array_size=latent, encoded_feature_size=latent,
                hidden_sizes=[16], out_size=latent, n_heads=2, rngs=rngs,
            )
        ],
        n_steps=10,
    ),
    decoder=MLPEquivariantDecoder(
        in_graph_structure=in_structure, in_array_size=latent, encoded_feature_size=latent,
        hidden_sizes=[16], activation=nnx.leaky_relu, out_structure=out_structure, rngs=rngs,
    ),
)
print("trainable parameters:", n_params(custom_model))

trainable parameters: 6435


## 4. Training

Finally, let us train the model on the problem, with `energnn.trainer`.

A `Trainer` takes:
- a model;
- an [optax](https://optax.readthedocs.io/) optimizer;
- a problem loader for training;
- optionally, another problem loader for validation;
- an [orbax](https://orbax.readthedocs.io/) checkpoint manager;
- a metrics **tracker** (MLflow is provided, here a minimal in-memory one).

In [34]:
import tempfile

import flatdict
import optax
import orbax.checkpoint as ocp

from energnn.tracker import Tracker
from energnn.trainer import Trainer


class MemoryTracker(Tracker):
    "Keeps every logged metric in a list of flat records."

    def __init__(self):
        self.records = []

    def init_run(self, *, name, tags, cfg):
        pass

    def stop_run(self):
        pass

    def run_append(self, *, metrics, step):
        flat = flatdict.FlatDict(metrics, delimiter="/")
        self.records.append({"step": step, **{k: float(np.mean(v)) for k, v in flat.items() if np.size(v) > 0}})


trainer = Trainer(model=model, gradient_transformation=optax.adam(learning_rate=1e-2))
checkpoint_manager = ocp.CheckpointManager(tempfile.mkdtemp(prefix="energnn_webinar_"),
                                           options=ocp.CheckpointManagerOptions(max_to_keep=3))
tracker = MemoryTracker()

Before training, on a validation problem: the untrained model asks for arbitrary setpoint changes.

In [35]:
test_problem = val_loader.problems[0]
test_context = test_problem.context.to_backend(JaxBackend())


def voltages_with(model):
    "The test network's voltages once the model's decision is applied, and the score."
    model.eval()
    decision, _ = model(test_context)
    v, delta_v = simulate(test_problem.select(), decision.generator.delta_v)
    return v, float(voltage_objective(v, delta_v))


v_untrained, score = voltages_with(model)
print(f"untrained model: score {score:.0f}")

untrained model: score 77199


In [36]:
best_score = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=15,
    checkpoint_manager=checkpoint_manager,
    tracker=tracker,
    log_period=4,
    eval_before_training=True,
)
print(f"best validation score: {best_score:.2f}")

Validation:   0%|          | 0/4 [00:00<?, ?batch/s]

ERROR:powsybl:14 buses have a voltage magnitude out of range [0.5, 2.0]: {VL-346_0=0.155172225157339, VL-920_0=0.2031302442002881, VL-1151_0=0.4874094566074859, VL-1662_0=0.20394280965923078, VL-1980_0=0.16801837426824415, VL-2019_0=0.18941150805425053, VL-3184_0=0.15170655407422703, VL-3601_0=0.15114239935698529, VL-4313_0=0.18369503936974393, VL-4339_1=0.20325487819083207, VL-4728_0=0.15114131475928616, VL-5174_0=0.19195966070311613, VL-8542_0=0.19193812235684074, VL-9119_0=0.4938618967509346}
ERROR:powsybl:14 buses have a voltage magnitude out of range [0.5, 2.0]: {VL-346_0=0.14677877514714127, VL-920_0=0.1907294012306832, VL-1151_0=0.47193982886521285, VL-1662_0=0.19173997660913175, VL-1980_0=0.1416212079547805, VL-2019_0=0.17351237678435152, VL-3184_0=0.1431412947172414, VL-3601_0=0.14261248850307587, VL-4313_0=0.16436315802283766, VL-4339_1=0.19088730568985499, VL-4728_0=0.14261146511600808, VL-5174_0=0.17610807418830549, VL-8542_0=0.1760883144290422, VL-9119_0=0.4786890827590521

Validation:  25%|██▌       | 1/4 [00:01<00:05,  1.93s/batch, score=5.5603e+02]

ERROR:powsybl:14 buses have a voltage magnitude out of range [0.5, 2.0]: {VL-346_0=0.15564511258624408, VL-920_0=0.19567578631833307, VL-1151_0=0.4705866760831047, VL-1662_0=0.19662251622724128, VL-1980_0=0.15730488377875623, VL-2019_0=0.17621514279927977, VL-3184_0=0.15155734180604621, VL-3601_0=0.15099587199279216, VL-4313_0=0.16853898668902142, VL-4339_1=0.19580679602196843, VL-4728_0=0.15099478844657327, VL-5174_0=0.17988262272861116, VL-8542_0=0.17986243945571387, VL-9119_0=0.4773307060879972}


Validation:  50%|█████     | 2/4 [00:03<00:03,  1.83s/batch, score=5.5517e+02]

ERROR:powsybl:14 buses have a voltage magnitude out of range [0.5, 2.0]: {VL-346_0=0.16649338045847226, VL-920_0=0.20061704792922494, VL-1151_0=0.4810433963461003, VL-1662_0=0.20156610346507084, VL-1980_0=0.16546385862564392, VL-2019_0=0.18439549283744872, VL-3184_0=0.16346977407928176, VL-3601_0=0.1629100412178899, VL-4313_0=0.17586840432705303, VL-4339_1=0.2007931417220529, VL-4728_0=0.16290887217560515, VL-5174_0=0.18688752400696004, VL-8542_0=0.18686655476690275, VL-9119_0=0.48730678759820084}
ERROR:powsybl:12 buses have a voltage magnitude out of range [0.5, 2.0]: {VL-346_0=0.19283194543740761, VL-920_0=0.22054035670746122, VL-1662_0=0.2213505656145034, VL-1980_0=0.19171000609586375, VL-2019_0=0.2064915051787555, VL-3184_0=0.18938287430110992, VL-3601_0=0.18897501786071055, VL-4313_0=0.2017387692085153, VL-4339_1=0.22067969568680057, VL-4728_0=0.1889736617761808, VL-5174_0=0.20935381125375888, VL-8542_0=0.2093303212409909}
ERROR:powsybl:14 buses have a voltage magnitude out of ran

Validation:  75%|███████▌  | 3/4 [00:05<00:02,  2.01s/batch, score=5.5366e+02]

ERROR:powsybl:14 buses have a voltage magnitude out of range [0.5, 2.0]: {VL-346_0=0.16541571926466764, VL-920_0=0.2059573482374745, VL-1151_0=0.48954673737663146, VL-1662_0=0.20683210993087975, VL-1980_0=0.17059609199365633, VL-2019_0=0.19053801742174525, VL-3184_0=0.16223605881185807, VL-3601_0=0.1617201212191427, VL-4313_0=0.18190292044401954, VL-4339_1=0.20612585612489087, VL-4728_0=0.1617189607157226, VL-5174_0=0.19302451810422128, VL-8542_0=0.19300286027832528, VL-9119_0=0.4964104095843934}


Validation: 100%|██████████| 4/4 [00:02<00:00,  1.48batch/s, score=3.9211e+00]
best validation score: 3.92


In [37]:
curve = pd.DataFrame(tracker.records).groupby("step").first()["eval/score"].dropna()

fig = go.Figure(go.Scatter(x=curve.index, y=curve.values, mode="lines+markers", line={"color": "#1f8ff0"}))
fig.update_layout(xaxis_title="training step", yaxis_title="validation score", yaxis_type="log", width=640, height=320,
                  margin={"l": 60, "r": 20, "t": 20, "b": 50}, plot_bgcolor="white")

After training, on the same unseen problem: one forward pass does what the gradient descent of section 2 did with a power flow and an adjoint per step.

In [38]:
v_trained, score = voltages_with(model)
print(f"trained model: score {score:.2f}")
voltage_histogram(
    {"reference setpoints": test_problem.context.bus.v_mag, "untrained model setpoints": v_untrained,
     "trained model setpoints": v_trained})

trained model: score 3.38


The same map as in section 2, for this unseen problem.

**Before**, with the reference setpoints (undervoltages in blue, overvoltages in red):

In [39]:
print(n_violations(test_problem.context.bus.v_mag), "buses out of the band")
plot_voltages(test_problem.context, test_problem.select())

369 buses out of the band


**After**, with the setpoints of the trained model:

In [40]:
print(n_violations(v_trained), "buses out of the band")
plot_voltages(test_problem.context, test_problem.select(), v_mag=v_trained)

6 buses out of the band


**Checkpoints**: the three best are kept; a `Trainer` restores one in place (which is also how training resumes).

In [41]:
print("saved steps:", checkpoint_manager.all_steps(), "| best step:", checkpoint_manager.best_step())

restored = MediumRecurrentEquivariantGNN(in_structure=in_structure, out_structure=out_structure)
Trainer(model=restored, gradient_transformation=optax.adam(1e-2)).load_checkpoint(checkpoint_manager, best=True)
print(f"restored model: score {voltages_with(restored)[1]:.2f}")

saved steps: [208, 224, 240] | best step: 240
restored model: score 3.38


## Wrap-up

- **Data** (`energnn.graph`): classes of objects with ports and features, glued by addresses; converters, padding, batching.
- **Problem** (`energnn.problem`): contexts, decisions, a score and a gradient (labels are one way to get a gradient, not the only one).
- **Models** (`energnn.model`): normalizer, encoder, coupler, decoder; ready-to-use or assembled by hand.
- **Training** (`energnn.trainer`): `Trainer` + optax + orbax + tracker.

[energnn.readthedocs.io](https://energnn.readthedocs.io/) · [github.com/energnn/energnn](https://github.com/energnn/energnn)